In [0]:
#Setting the catalog and schema
# This command tells python spark to perform an sql
spark.sql("USE CATALOG shopperfomance")
spark.sql("USE SCHEMA shop_perfomances")

In [0]:
#Loading my Customers table
customers_df = spark.table("customers").toPandas()

#Checking the structure
customers_df.info()

#Checking my table
display(customers_df)

#Checking my unique Customer Segments
customers_df["CustomerSegment"].unique()

#Checking my Customer Segment count
customers_df["CustomerSegment"].value_counts()

#Checking my unique Cities - Tehran vs tehran,Mashhad vs Mashad, and None
customers_df["City"].unique()

#Checking my Customer Segment count by City
customers_df.groupby(["City", "CustomerSegment"]).size()

#Checking my missing values --- found
customers_df.isnull().sum()

#Checking my missing Age by Customer Segment
customers_df.loc[customers_df["Age"].isnull(), "CustomerSegment"].value_counts()

#Checking my duplicate rows -- 0 duplicates
customers_df.duplicated().sum()

#Checking my duplicate Customer IDs
customers_df["CustomerID"].duplicated().sum()

#Checking my Age range
customers_df["Age"].min(), customers_df["Age"].max()

#Checking my Age summary
customers_df["Age"].describe()

#Checking my Age by City
customers_df.groupby("City")["Age"].agg(["min", "max", "mean", "count"])


In [0]:
#Cleaning my Customers table

#Fixing my City names to one format
customers_df["City"] = customers_df["City"].replace({
    "tehran": "Tehran",
    "Mashad": "Mashhad"
})
#Checking my missing City values after fixing
customers_df["City"].isnull().sum()

#Filling my missing City values
customers_df["City"] = customers_df["City"].fillna("Unknown")

#Checking my missing City values after cleaning
customers_df["City"].isnull().sum()

#Cleaning my missing Age values
customers_df["Age"] = customers_df["Age"].fillna(customers_df["Age"].median())

#Checking my missing Age values after cleaning
customers_df["Age"].isnull().sum()


In [0]:
#Loading my Orders table
orders_df = spark.table("orders").toPandas()

#Checking the structure
orders_df.info()

#Checking my table
display(orders_df)

#Checking my unique Payment Methods
orders_df["PaymentMethod"].unique()

#Checking my unique Order Status
orders_df["Status"].unique()

#Checking my missing values -- found
orders_df.isnull().sum()

#Checking status of orders with missing quantities
orders_df[orders_df["Quantity"].isna()]["Status"].value_counts()

#Checking cancelled orders with missing quantities
filtered_df = orders_df[(orders_df["Status"] == "Cancelled") & (orders_df["Quantity"].isna())]
display(filtered_df)

#Checking my duplicate rows - 0 duplicate
orders_df.duplicated().sum()

#Checking my Order IDs
orders_df["OrderID"].duplicated().sum()

#Checking my duplicate Order IDs
display(orders_df[orders_df["OrderID"].duplicated(keep=False)].sort_values("OrderID").head(10))

#Checking my Quantity range
orders_df["Quantity"].min(), orders_df["Quantity"].max()

#Checking my Discount range
orders_df["Discount"].min(), orders_df["Discount"].max()

#Checking my orders with negative Quantity
display(orders_df[orders_df["Quantity"] < 0])

#Checking the Status of my negative Quantity orders --- Locate the rows where Quantity is less than 0, and give me the Status column
orders_df.loc[orders_df["Quantity"] < 0, "Status"].value_counts()

#Checking my negative Quantity by Product
orders_df.loc[orders_df["Quantity"] < 0, "ProductID"].value_counts()

#Checking my negative Quantity by Discount
orders_df.loc[orders_df["Quantity"] < 0, "Discount"].value_counts().sort_index()

#Checking my negative Quantity by Payment Method
orders_df.loc[orders_df["Quantity"] < 0, "PaymentMethod"].value_counts()

#Checking my Orders summary
orders_df[["Quantity", "Discount"]].describe()

#Checking my missing Discount values
display(orders_df[orders_df["Discount"].isnull()])

In [0]:
#Cleaning my Orders table

#Removing my duplicate Order records
orders_df = orders_df.drop_duplicates()

#Checking my duplicate rows after cleaning
orders_df.duplicated().sum()

#Checking my negative Quantity after removing duplicates
orders_df.loc[orders_df["Quantity"] < 0, "Status"].value_counts()

#Checking my negative Quantity records
display(orders_df[orders_df["Quantity"] < 0].head(10))

#Cleaning my missing Discount values
orders_df["Discount"] = orders_df["Discount"].fillna(0)

#Checking my missing Discount values after cleaning
orders_df["Discount"].isnull().sum()

In [0]:
#Loading my Products table
products_df = spark.table("products").toPandas()

#Checking the structure
products_df.info()

#Checking my unique Product Names
products_df["ProductName"].unique()

#Checking my unique Categories
products_df["Category"].unique()

#Checking my missing values
products_df.isnull().sum()

#Checking my duplicate rows - 0 duplicates
products_df.duplicated().sum()

#Checking my Product IDs
products_df["ProductID"].duplicated().sum()

#Checking my Unit Price range
products_df["UnitPrice"].min(), products_df["UnitPrice"].max()

#Checking my Unit Price summary
products_df["UnitPrice"].describe()

#Checking unit price for each product
unit_price = (
    products_df
    .groupby(["ProductID", "ProductName"])["UnitPrice"]
    .first()
    .reset_index()
)
display(unit_price)




In [0]:
#Loading my Payments table
payments_df = spark.table("payments").toPandas()

#Checking the structure
payments_df.info()

#Checking my table
display(payments_df)

#Checking my unique Payment Status
payments_df["PaymentStatus"].unique()

#Checking my missing value---- found - payment dates
payments_df.isnull().sum()

#Checking my duplicate rows - 0 duplicates
payments_df.duplicated().sum()

#Checking my Payment IDs
payments_df["PaymentID"].duplicated().sum()

#Checking my Payments summary
payments_df.describe()

#Checking my Payment columns
payments_df.columns

#Checking my Payment Date data type
payments_df["PaymentDate"].dtype

#Checking my missing Payment Dates
payments_df["PaymentDate"].isnull().sum()

#Checking my Payment Date value types
payments_df["PaymentDate"].apply(type).value_counts()

#Checking my missing Payment Dates
display(payments_df[payments_df["PaymentDate"].isnull()])

#Checking my Payment Status for missing Payment Dates
payments_df.loc[payments_df["PaymentDate"].isnull(), "PaymentStatus"].value_counts()



In [0]:
#Cleaning my Payments table

#Keeping missing PaymentDate values as missing
#35 PaymentDate values are missing and cannot be reliably replaced


In [0]:
#Checking my table relationships

#Checking if my Customer IDs in Orders exist in Customers --- 30 transactions dont exist in customer table
orders_df["CustomerID"].isin(customers_df["CustomerID"]).value_counts()

#Checking my Orders with missing Customer IDs that does not exist in the Customers table are labelled 99999
display(orders_df[~orders_df["CustomerID"].isin(customers_df["CustomerID"])])

#Checking if my Order IDs in Payments exist in Orders
payments_df["OrderID"].isin(orders_df["OrderID"]).value_counts()

#Checking if my Product IDs in Orders exist in Products
orders_df["ProductID"].isin(products_df["ProductID"]).value_counts()


In [0]:
#Final Data Validation - /n means new line.

#Customers
print("CUSTOMERS")
print("Duplicate rows:", customers_df.duplicated().sum())
print("Duplicate Customer IDs:", customers_df["CustomerID"].duplicated().sum())
print("Missing values:")
print(customers_df.isnull().sum())

#Orders
print("\nORDERS")
print("Duplicate rows:", orders_df.duplicated().sum())
print("Duplicate Order IDs:", orders_df["OrderID"].duplicated().sum())
print("Missing Discount:", orders_df["Discount"].isnull().sum())

#Products
print("\nPRODUCTS")
print("Duplicate rows:", products_df.duplicated().sum())
print("Duplicate Product IDs:", products_df["ProductID"].duplicated().sum())

#Payments
print("\nPAYMENTS")
print("Duplicate rows:", payments_df.duplicated().sum())
print("Duplicate Payment IDs:", payments_df["PaymentID"].duplicated().sum())
print("Missing Payment Dates:", payments_df["PaymentDate"].isnull().sum())